# Dota 2 RAG Pipeline (v1)

Монолитный RAG-пайплайн без роутинга: все запросы идут через три коллекции одновременно.  
**Следующий шаг → `langgraph_node_agent.ipynb`** — v2 с маршрутизацией и LangGraph.

**Зависимость:** корпусы должны быть созданы в `data_preprocessing.ipynb`.

---

## Структура

| Секция | Что делаем |
|--------|-----------|
| Импорты | Все зависимости |
| BGE-M3 | Загрузка embedding-модели (1024-dim, без префиксов) |
| ChromaDB | Инициализация + индексация трёх коллекций |
| Поиск | `search_all_collections` → `get_top_contexts` |
| Reranker | Загрузка BGE Reranker v2-m3 + `rerank_results` |
| Llama 3.2-3B | Загрузка bfloat16 + `generate_with_llama` |
| Pipeline | `ask_dota2_with_llm` — полный пайплайн |
| Чат | `dota2_chat` — интерактивный режим |

## Коллекции ChromaDB

| Коллекция | Документов |
|-----------|-----------|
| `dota2_heroes` | ~1 400 |
| `dota2_items` | ~475 |
| `dota2_patches` | ~340 |


In [1]:
import json
import os
import time
from pathlib import Path
from typing import Dict, List, Optional, Any

import torch
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoTokenizer, AutoModelForCausalLM
import chromadb
from chromadb.config import Settings
from tqdm.auto import tqdm
from dotenv import load_dotenv

load_dotenv()


c:\python\dota_ai\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
# BAAI/bge-m3: мультиязычная модель, 1024-dim, до 8192 токенов
# Не требует префиксов "query:" / "passage:" — в отличие от E5-семейства
MODEL_PATH = Path("models/bge-m3")

if MODEL_PATH.exists():
    embedding_model = SentenceTransformer(str(MODEL_PATH))
else:
    print("Скачивание BAAI/bge-m3 (~2.3 ГБ)...")
    embedding_model = SentenceTransformer("BAAI/bge-m3")
    MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
    embedding_model.save(str(MODEL_PATH))

dim = embedding_model.get_sentence_embedding_dimension()
print(f"BGE-M3 готов, dim={dim}, max_seq={embedding_model.max_seq_length}")

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 1817.24it/s, Materializing param=pooler.dense.weight]                               


BGE-M3 готов, dim=1024, max_seq=8192


## ChromaDB — векторное хранилище

Персистентный клиент сохраняет данные в `data/chromadb/`.  
Три коллекции: `dota2_heroes`, `dota2_items`, `dota2_patches`.

In [3]:
db_path = Path("data/chromadb")
db_path.mkdir(parents=True, exist_ok=True)

client = chromadb.PersistentClient(
    path=str(db_path),
    settings=Settings(anonymized_telemetry=False, allow_reset=True)
)

collections = [c.name for c in client.list_collections()]
print(f"ChromaDB готов, коллекций {len(collections)}: {collections}")

ChromaDB готов, коллекций 3: ['dota2_patches', 'dota2_heroes', 'dota2_items']


## Индексация корпусов

`index_corpus` — универсальная функция: загружает JSONL → батчами генерирует эмбеддинги → сохраняет в ChromaDB.  
`clean_metadata` нужна, потому что ChromaDB не принимает списки в metadata — конвертируем в строки.

In [4]:
def clean_metadata(metadata: dict) -> dict:
    """Преобразует списки в metadata в строки."""
    cleaned = {}
    for key, value in metadata.items():
        if isinstance(value, list):
            # Преобразуем список в строку через запятую
            cleaned[key] = ', '.join(str(v) for v in value)
        elif isinstance(value, (str, int, float, bool)) or value is None:
            cleaned[key] = value
        else:
            # Для любых других типов преобразуем в строку
            cleaned[key] = str(value)
    return cleaned

In [5]:
def index_corpus(
    corpus_file: Path,
    collection_name: str,
    embedding_model: SentenceTransformer,
    client: chromadb.PersistentClient,
    batch_size: int = 32,
    reset: bool = False,
) -> chromadb.Collection:
    """
    Индексирует JSONL-корпус в ChromaDB коллекцию.

    Args:
        corpus_file: Путь к .jsonl файлу
        collection_name: Имя коллекции
        embedding_model: BGE-M3 (без префиксов)
        client: ChromaDB PersistentClient
        batch_size: Размер батча для encode()
        reset: True — удалит и пересоздаст существующую коллекцию

    Returns:
        Готовая ChromaDB коллекция
    """
    documents = []
    with open(corpus_file, "r", encoding="utf-8") as f:
        for line in f:
            doc = json.loads(line)
            if "metadata" in doc:
                doc["metadata"] = clean_metadata(doc["metadata"])
            documents.append(doc)

    if reset and collection_name in [c.name for c in client.list_collections()]:
        client.delete_collection(collection_name)

    collection = client.get_or_create_collection(
        name=collection_name,
        metadata={"hnsw:space": "cosine"},
    )

    for i in tqdm(range(0, len(documents), batch_size), desc=collection_name):
        batch = documents[i : i + batch_size]
        embeddings = embedding_model.encode(
            [d["text"] for d in batch],
            show_progress_bar=False,
            convert_to_numpy=True,
        )
        collection.add(
            ids=[d["id"] for d in batch],
            embeddings=embeddings.tolist(),
            documents=[d["text"] for d in batch],
            metadatas=[d.get("metadata", {}) for d in batch],
        )

    print(f"{collection_name}: {collection.count()} документов")
    return collection


## 🎯 Индексация корпуса Heroes (герои)

In [60]:
heroes_collection = index_corpus(
    corpus_file=Path("data/clean/heroes_corpus.jsonl"),
    collection_name="dota2_heroes",
    embedding_model=embedding_model,
    client=client,
    batch_size=32,
    reset=True,
)

dota2_heroes: 100%|██████████| 40/40 [00:35<00:00,  1.13it/s]

dota2_heroes: 1268 документов


## 🛡️ Индексация корпуса Items (предметы)

In [61]:
items_collection = index_corpus(
    corpus_file=Path("data/clean/items_corpus.jsonl"),
    collection_name="dota2_items",
    embedding_model=embedding_model,
    client=client,
    batch_size=32,
    reset=True,
)

dota2_items: 100%|██████████| 15/15 [01:01<00:00,  4.12s/it]

dota2_items: 475 документов


## 📝 Индексация корпуса Patches (заметки о патчах)

In [62]:
patches_collection = index_corpus(
    corpus_file=Path("data/clean/patches_corpus.jsonl"),
    collection_name="dota2_patches",
    embedding_model=embedding_model,
    client=client,
    batch_size=32,
    reset=True,
)

dota2_patches: 100%|██████████| 11/11 [00:36<00:00,  3.36s/it]

dota2_patches: 327 документов


In [6]:
def search_all_collections(
    query: str,
    top_k: int = 5,
    collections_to_search: Optional[List[str]] = None,
) -> dict:
    """
    Поиск по всем коллекциям с объединением результатов.

    Args:
        query: Поисковый запрос
        top_k: Число результатов из каждой коллекции
        collections_to_search: Список имён коллекций (по умолчанию все три)

    Returns:
        Словарь {collection_name: {documents, metadatas, distances, similarities}}
    """
    if collections_to_search is None:
        collections_to_search = ["dota2_heroes", "dota2_items", "dota2_patches"]

    query_embedding = embedding_model.encode(query, convert_to_numpy=True)

    all_results = {}
    for collection_name in collections_to_search:
        try:
            collection = client.get_collection(collection_name)
            results = collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k,
                include=["documents", "metadatas", "distances"],
            )
            all_results[collection_name] = {
                "documents": results["documents"][0],
                "metadatas": results["metadatas"][0],
                "distances": results["distances"][0],
                "similarities": [1 - d for d in results["distances"][0]],
            }
        except Exception as e:
            print(f"{collection_name}: {e}")
            all_results[collection_name] = None

    return all_results

In [7]:
query = "Как работает Anti-Mage против магов?"
results = search_all_collections(query, top_k=3)

print(f"🔍 {query}\n")
for coll, res in results.items():
    if res is None:
        continue
    best_sim = res["similarities"][0]
    preview = res["documents"][0][:120].replace("\n", " ")
    print(f"{coll}  sim={best_sim:.3f}, {preview}...")

🔍 Как работает Anti-Mage против магов?

dota2_heroes  sim=0.643, Герой: Anti-Mage  Описание: Если Anti-Mage наберёт полную силу, мало кто сможет его остановить. Он способен забирать у в...
dota2_items  sim=0.504, Предмет: Mage Slayer  Способности: Пассивное: Mage SlayerАтакуемые враги получают 20 урона в секунду и наносят на 40% ме...
dota2_patches  sim=0.570, Патч 7.40 — Anti-Mage  Общие изменения:   • Способность Counterspell Ally удалена из игры  Таланты:   • Талант 15 уровня...



## 🔁 Reranker: BAAI/bge-reranker-v2-m3

**Зачем reranker?**  
Bi-encoder (BGE-M3) ищет быстро, но оценивает запрос и документ раздельно → неточности.  
Cross-encoder видит запрос **и** документ одновременно → гораздо точнее, но медленнее.  

**Стратегия:**  
1. `search_all_collections(top_k=10)` — быстро достаём 30 кандидатов (10 × 3 коллекции)  
2. `rerank_results(candidates, top_n=5)` — cross-encoder переоценивает и выбирает лучшие 5  

**Модель:** `BAAI/bge-reranker-v2-m3` (~570 МБ, мультиязычная, включая русский)


In [7]:
RERANKER_PATH = Path("models/bge-reranker-v2-m3")
RERANKER_NAME = "BAAI/bge-reranker-v2-m3"

if RERANKER_PATH.exists() and (RERANKER_PATH / "config.json").exists():
    reranker = CrossEncoder(str(RERANKER_PATH), max_length=512)
else:
    print(f"Скачивание {RERANKER_NAME}")
    reranker = CrossEncoder(RERANKER_NAME, max_length=512)
    RERANKER_PATH.mkdir(parents=True, exist_ok=True)
    reranker.save(str(RERANKER_PATH))

if torch.cuda.is_available():
    reranker.model.to("cuda")

device = next(reranker.model.parameters()).device
print(f"BGE Reranker v2-m3, device={device}, max_length=512")

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 1226.68it/s, Materializing param=roberta.encoder.layer.23.output.dense.weight]              


BGE Reranker v2-m3, device=cuda:0, max_length=512


In [8]:
def rerank_results(query: str, candidates: list, top_n: int = 5) -> list:
    """
    Переоценивает кандидатов с помощью cross-encoder reranker.

    Стратегия двухэтапного поиска (bi-encoder + cross-encoder):
      1. Bi-encoder (BGE-M3) быстро отбирает кандидатов из ChromaDB
      2. Cross-encoder (BGE Reranker v2-m3) точно переоценивает каждую пару [запрос, документ]

    Args:
        query: Поисковый запрос
        candidates: Результаты от get_top_contexts
        top_n: Сколько лучших вернуть после reranking

    Returns:
        Топ-N кандидатов, отсортированных по rerank_score (убывание)
    """
    if not candidates:
        return []

    pairs = [(query, c["text"]) for c in candidates]
    scores = reranker.predict(pairs, show_progress_bar=False)

    reranked = []
    for cand, score in zip(candidates, scores):
        item = dict(cand)
        item["rerank_score"] = float(score)
        item["bi_encoder_score"] = cand["similarity"]
        reranked.append(item)

    reranked.sort(key=lambda x: x["rerank_score"], reverse=True)
    return reranked[:top_n]

In [9]:
def get_top_contexts(results: dict, top_n: int = 5, min_similarity: float = 0.5) -> list:
    """
    Извлекает топ-N наиболее релевантных документов из всех коллекций.

    Args:
        results: Результаты от search_all_collections
        top_n: Количество лучших результатов
        min_similarity: Минимальная similarity для включения

    Returns:
        Список словарей с ключами: text, similarity, collection, metadata
    """
    all_docs = []
    for collection_name, collection_results in results.items():
        if collection_results is None:
            continue
        for doc, metadata, similarity in zip(
            collection_results["documents"],
            collection_results["metadatas"],
            collection_results["similarities"],
        ):
            if similarity >= min_similarity:
                all_docs.append({
                    "text": doc,
                    "similarity": similarity,
                    "collection": collection_name,
                    "metadata": metadata,
                })

    all_docs.sort(key=lambda x: x["similarity"], reverse=True)
    return all_docs[:top_n]

In [10]:
_SOURCE_LABELS = {
    "dota2_heroes": "База героев",
    "dota2_items": "База предметов",
    "dota2_patches": "История патчей",
}


def format_context_for_llm(contexts: list) -> str:
    """Форматирует список контекстов в текстовый блок для LLM."""
    if not contexts:
        return "Контекст не найден."
    parts = []
    for i, ctx in enumerate(contexts, 1):
        source = _SOURCE_LABELS.get(ctx["collection"], ctx["collection"])
        parts.append(f"[Контекст {i}] {source} (sim={ctx['similarity']:.2%})")
        parts.append(ctx["text"])
        parts.append("")
    return "\n".join(parts)


def create_rag_messages(query: str, contexts: list) -> list:
    """
    Строит список messages для apply_chat_template: [system, user].

    Разделение на system/user важно для Llama 3.1 Instruct:
    - system: инструкции + контекст
    - user: только вопрос
    """
    context_text = format_context_for_llm(contexts)
    system = (
        "Ты — эксперт по Dota 2. Отвечай строго на основе предоставленного контекста.\n\n"
        "ПРАВИЛА:\n"
        "1. Используй ТОЛЬКО информацию из контекста ниже\n"
        "2. Если данных недостаточно — скажи об этом прямо\n"
        "3. Приводи конкретные цифры и факты из контекста\n"
        "4. Отвечай на русском языке\n"
        "5. Начинай ответ сразу — без вводных слов и комментариев к контексту\n\n"
        f"КОНТЕКСТ:\n{context_text}"
    )
    return [
        {"role": "system", "content": system},
        {"role": "user", "content": query},
    ]



## 🔬 Тест реранкера: сравнение bi-encoder vs cross-encoder

Запускаем поиск и смотрим как reranker меняет порядок документов.


In [11]:
def test_reranker(query: str, candidates_per_collection: int = 5, top_n: int = 5):
    """
    Показывает документы до и после reranking для наглядного сравнения.
    """
    SEP = "─" * 80

    print(f"\n{'═' * 80}")
    print(f"Запрос: {query}")
    print(f"{'═' * 80}")

    # ── Шаг 1: Bi-encoder поиск ────────────────────────────────────────────
    results = search_all_collections(query, top_k=candidates_per_collection)
    candidates = get_top_contexts(
        results,
        top_n=candidates_per_collection * 3,
        min_similarity=0.0,   # без фильтра, хотим видеть всё
    )


    print(f"\n{'BI-ENCODER':^80}")
    print(f"Получено кандидатов: {len(candidates)}")
    print(SEP)
    for rank, c in enumerate(candidates, 1):
        preview = c["text"][:140].replace("\n", " ")
        if len(c["text"]) > 140:
            preview += "…"
        print(f"  #{rank:>2}, bi={c['similarity']:.4f}, {c['collection']}")
        print(f"      {preview}")
        print()

    # ── Шаг 2: Cross-encoder reranking ─────────────────────────────────────
    reranked = rerank_results(query, candidates, top_n=top_n)

    print(f"\n{'ПОСЛЕ RERANKING (топ ' + str(top_n) + ')':^80}")
    print(SEP)

    # Строим словарь старых позиций по тексту — для отображения Δ
    old_ranks = {c["text"]: i + 1 for i, c in enumerate(candidates)}

    for new_rank, c in enumerate(reranked, 1):
        old_rank = old_ranks.get(c["text"], "?")
        delta = old_rank - new_rank if isinstance(old_rank, int) else 0
        arrow = (f"↑{delta}"  if delta > 0
                 else f"↓{abs(delta)}" if delta < 0
                 else "=")
        preview = c["text"][:140].replace("\n", " ")
        if len(c["text"]) > 140:
            preview += "…"
        print(f"  #{new_rank:>2}, rerank={c['rerank_score']:+.4f}, bi={c['bi_encoder_score']:.4f}, "
              f"({arrow} с #{old_rank}), {c['collection']}")
        print(f"      {preview}")
        print()

    return reranked


In [ ]:
# Тест 1
test_reranker("Расскажи про способности Anti-Mage", candidates_per_collection=5, top_n=5)


════════════════════════════════════════════════════════════════════════════════
Запрос: Расскажи про способности Anti-Mage
════════════════════════════════════════════════════════════════════════════════

                                   BI-ENCODER                                   
Получено кандидатов: 11
────────────────────────────────────────────────────────────────────────────────
  # 1, bi=0.3915, dota2_heroes
      Герой: Anti-Mage  Описание: Если Anti-Mage наберёт полную силу, мало кто сможет его остановить. Он способен забирать у врагов ману каждым уд…

  # 2, bi=0.2735, dota2_heroes
      Герой: Anti-Mage  Способность: Counterspell Макс. уровень: 4  Пассивно увеличивает сопротивление магическому урону. Можно применить, чтобы с…

  # 3, bi=0.2322, dota2_heroes
      Герой: Anti-Mage  ТАЛАНТЫ:  Уровень 10:   +3 здоровья в секунду   +500 к радиусу Mana Void  Уровень 15:   +1.6% к сжигаемой макс. мане от Ma…

  # 4, bi=0.2281, dota2_heroes
      Герой: Anti-Mage  Способность:

[{'text': 'Герой: Anti-Mage\n\nОписание: Если Anti-Mage наберёт полную силу, мало кто сможет его остановить. Он способен забирать у врагов ману каждым ударом или телепортироваться на небольшие расстояния, что не позволяет врагам загнать его в угол.\n\nОсновной атрибут: Ловкость\nТип атаки: Ближний бой\nСложность: Простой\n\nБАЗОВЫЕ ХАРАКТЕРИСТИКИ:\n  Здоровье: 582\n  Мана: 219\n  Броня: 5\n  Урон: 53-57\n  Скорость атаки: 1.4\n  Дальность атаки: 150\n  Скорость передвижения: 310\n  Магическое сопротивление: 25%\n\nРОЛИ В ИГРЕ:\n  Основа (Carry): 3/3 - отлично подходит\n  Поддержка (Support): 0/3 - не подходит\n  Быстрый урон (Nuker): 1/3 - слабо подходит\n  Контроль (Disabler): 0/3 - не подходит\n  Лес (Jungler): 0/3 - не подходит\n  Стойкость (Durable): 0/3 - не подходит\n  Побег (Escape): 3/3 - отлично подходит\n  Осада (Pusher): 0/3 - не подходит\n  Инициация (Initiator): 0/3 - не подходит\n\nАТРИБУТЫ:\n  Сила: 21 + 1.6 за уровень\n  Ловкость: 24 + 2.8 за уровень\n  Интеллект: 12 + 

In [ ]:
# Тест 2
test_reranker("Какие предметы дают невидимость?", candidates_per_collection=5, top_n=5)


════════════════════════════════════════════════════════════════════════════════
Запрос: Какие предметы дают невидимость?
════════════════════════════════════════════════════════════════════════════════

                                   BI-ENCODER                                   
Получено кандидатов: 12
────────────────────────────────────────────────────────────────────────────────
  # 1, bi=0.3196, dota2_items
      Предмет: Gem of True Sight  Способности: Активное: Reveal Раскрывает невидимость в радиусе 300, показывая существ и варды даже в тумане войн…

  # 2, bi=0.2632, dota2_items
      Предмет: Sentry Ward  Способности: Использование: Plant Устанавливает невидимый тотем всевидения, раскрывающий невидимость вражеских существ…

  # 3, bi=0.1973, dota2_items
      Предмет: Shadow Amulet  Способности: Активное: Fade Накладывает на владельца или выбранного союзника невидимость, которая действует 3.5 сек.…

  # 4, bi=0.1517, dota2_heroes
      Герой: Visage  Способность: Silent 

[{'text': 'Предмет: Gem of True Sight\n\nСпособности:\nАктивное: Reveal Раскрывает невидимость в радиусе 300, показывая существ и варды даже в тумане войны.\nПассивное: True Sight Даёт владельцу и его союзникам возможность видеть невидимых существ и тотемы в радиусе 900.\nПассивное: EverlastingПредмет выпадает при смерти и не может быть уничтожен.',
  'similarity': 0.31964921951293945,
  'collection': 'dota2_items',
  'metadata': {'slug': 'gem-of-true-sight',
   'language': 'ru',
   'item_name': 'Gem of True Sight',
   'item_id': 30},
  'rerank_score': 0.8040882349014282,
  'bi_encoder_score': 0.31964921951293945},
 {'text': 'Предмет: Sentry Ward\n\nСпособности:\nИспользование: Plant Устанавливает невидимый тотем всевидения, раскрывающий невидимость вражеских существ и вардов в радиусе 1050, если они в зоне вашей видимости.\nДействует 7 мин.\nНе даёт наземный обзор.\nМожно передать один вард союзному герою, зажав клавишу Ctrl.',
  'similarity': 0.2632101774215698,
  'collection': 'dota

In [ ]:
# Тест 3
test_reranker("Что изменилось у Invoker в последних патчах?", candidates_per_collection=5, top_n=5)


════════════════════════════════════════════════════════════════════════════════
Запрос: Что изменилось у Invoker в последних патчах?
════════════════════════════════════════════════════════════════════════════════

                                   BI-ENCODER                                   
Получено кандидатов: 10
────────────────────────────────────────────────────────────────────────────────
  # 1, bi=0.3229, dota2_patches
      Патч 7.40 — Invoker  Таланты:   • Талант 15 уровня: уменьшение перезарядки Cold Snap усилено с 5 до 6 секунд   • Талант 20 уровня: бонус к у…

  # 2, bi=0.1939, dota2_heroes
      Герой: Invoker  Описание: Благодаря обширному арсеналу заклинаний Invoker может приспособиться к ходу любой битвы. Комбинируя три магичесих …

  # 3, bi=0.1912, dota2_heroes
      Герой: Invoker  Способность: Ghost Walk Макс. уровень: 1  Герой комбинирует стихии льда и молнии, становясь невидимым и ускоряя себе восстан…

  # 4, bi=0.1578, dota2_heroes
      Герой: Invoker  Спо

[{'text': 'Патч 7.40 — Invoker\n\nТаланты:\n  • Талант 15 уровня: уменьшение перезарядки Cold Snap усилено с 5 до 6 секунд\n  • Талант 20 уровня: бонус к урону и скорости атаки от Alacrity увеличен с 35 до 50\n\nAlacrity:\n  • Расход маны уменьшен с 90 до 75\n  • Дальность применения увеличена с 650 до 700\n\nIce Wall:\n  • Урон увеличен с [25 + 5 * уровень Exort] на [24 + 6 * уровень Exort]',
  'similarity': 0.3228696584701538,
  'collection': 'dota2_patches',
  'metadata': {'hero_name': 'Invoker',
   'hero_id': 74,
   'entity_type': 'hero',
   'patch_version': '7.40',
   'language': 'ru'},
  'rerank_score': 0.27734607458114624,
  'bi_encoder_score': 0.3228696584701538},
 {'text': 'Герой: Invoker\n\nОписание: Благодаря обширному арсеналу заклинаний Invoker может приспособиться к ходу любой битвы. Комбинируя три магичесих компонента, он может сотворить одно из десяти заклинаний, позволяющих уничтожить врага или спастись от него.\n\nОсновной атрибут: Интеллект\nТип атаки: Дальний бой\nС

## Llama 3.2-3B-Instruct (генерация)

Загрузка в bfloat16 (~6 GB VRAM) без квантования — не требует bitsandbytes.  
Работает на любой CUDA-совместимой видеокарте, включая Blackwell (sm_120).


## 🔐 Авторизация в Hugging Face

Загружаем токен из `.env` файла для доступа к Llama 3.1-8B.

In [12]:
from huggingface_hub import login, whoami

hf_token = os.getenv("HF_TOKEN")
if not hf_token:
    raise EnvironmentError("HF_TOKEN не найден. Добавьте в .env: HF_TOKEN=hf_...")

login(token=hf_token, add_to_git_credential=False)
user = whoami()
# print(f"HF авторизован: {user['name']}")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [12]:
LLAMA_MODEL_PATH = Path("models/llama3.2-3b")
LLAMA_MODEL_NAME = "meta-llama/Llama-3.2-3B-Instruct"

if LLAMA_MODEL_PATH.exists() and (LLAMA_MODEL_PATH / "config.json").exists():
    model_source = str(LLAMA_MODEL_PATH)
    use_token = False
    print(f"Модель найдена локально: {LLAMA_MODEL_PATH}")
else:
    model_source = LLAMA_MODEL_NAME
    use_token = True
    print(f"Будет скачана из HF: {LLAMA_MODEL_NAME}")
    LLAMA_MODEL_PATH.mkdir(parents=True, exist_ok=True)


Модель найдена локально: models\llama3.2-3b


In [13]:
import gc

gc.collect()
torch.cuda.empty_cache()

vram_free = (
    torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_reserved(0)
) / 1024**3
print(f"VRAM перед загрузкой: {vram_free:.1f} GB свободно")

# bfloat16: ~6 GB VRAM для 3B модели — без квантования, без bitsandbytes
llama_tokenizer = AutoTokenizer.from_pretrained(
    model_source,
    token=hf_token if use_token else None,
)

llama_model = AutoModelForCausalLM.from_pretrained(
    model_source,
    torch_dtype=torch.bfloat16,
    device_map={"": "cuda:0"},
    attn_implementation="sdpa",
    token=hf_token if use_token else None,
)
llama_model.eval()

# Сохраняем локально, если была загрузка с HF (чтобы не скачивать повторно)
if use_token:
    print(f"Сохранение в {LLAMA_MODEL_PATH} ...")
    llama_tokenizer.save_pretrained(str(LLAMA_MODEL_PATH))
    llama_model.save_pretrained(str(LLAMA_MODEL_PATH))
    print("Сохранено.")

vram_used = torch.cuda.memory_allocated(0) / 1024**3
vram_total = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"Llama 3.2-3B (bfloat16), VRAM: {vram_used:.1f} / {vram_total:.1f} GB")


VRAM перед загрузкой: 11.7 GB свободно


`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 254/254 [00:01<00:00, 127.83it/s, Materializing param=model.norm.weight]                              


Llama 3.2-3B (bfloat16), VRAM: 10.2 / 15.9 GB


In [14]:
def generate_with_llama(messages: list, max_new_tokens: int = 256, temperature: float = 0.3) -> str:
    """
    Генерирует ответ с помощью Llama 3.1-8B-Instruct.

    Args:
        messages: Список [{role, content}] для apply_chat_template.
        max_new_tokens: Максимальная длина ответа в токенах
        temperature: 0.0 = детерминированно, >0 = сэмплирование

    Returns:
        Сгенерированный текст без промпта и специальных токенов
    """
    formatted = llama_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    # max_length=2048: при NF4 (~7.4 GB) остаётся ~9 GB под KV-cache
    inputs = llama_tokenizer(
        formatted,
        return_tensors="pt",
        truncation=True,
        max_length=2048,
    ).to(llama_model.device)

    # Явно указываем eos_token_id для Llama3 — без этого модель иногда не останавливается
    eos_ids = [
        llama_tokenizer.eos_token_id,
        llama_tokenizer.convert_tokens_to_ids("<|eot_id|>"),
    ]

    with torch.inference_mode():
        outputs = llama_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature if temperature > 0 else None,
            do_sample=temperature > 0,
            top_p=0.9 if temperature > 0 else None,
            repetition_penalty=1.1,
            eos_token_id=eos_ids,
            pad_token_id=llama_tokenizer.eos_token_id,
            use_cache=True,
        )

    return llama_tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
    ).strip()


## 🎯 Полный RAG с Llama 3.2-3B

Объединяем поиск контекстов и генерацию ответов.


In [15]:
def ask_dota2_with_llm(
    query: str,
    top_k: int = 5,
    candidates_per_collection: int = 10,
    min_similarity: float = 0.3,
    max_tokens: int = 256,
    temperature: float = 0.3,
    show_contexts: bool = True,
) -> Optional[dict]:
    """
    Полный RAG-пайплайн: bi-encoder поиск → cross-encoder reranking → генерация.

    Args:
        query: Вопрос пользователя
        top_k: Число документов для LLM после reranking
        candidates_per_collection: Кандидатов из каждой коллекции для reranker
        min_similarity: Порог bi-encoder similarity для первичной фильтрации
        max_tokens: Максимальная длина ответа
        temperature: Температура генерации
        show_contexts: Вывести ли найденные контексты перед ответом
    """
    print(f"{query}\n")

    # 1. Bi-encoder: широкий поиск
    results = search_all_collections(query, top_k=candidates_per_collection)
    candidates = get_top_contexts(
        results, top_n=candidates_per_collection * 3, min_similarity=min_similarity
    )
    if not candidates:
        print("Кандидаты не найдены")
        return None

    # 2. Cross-encoder: точная переоценка
    contexts = rerank_results(query, candidates, top_n=top_k)

    if show_contexts:
        print("Контексты (после reranking):")
        for i, ctx in enumerate(contexts, 1):
            bi = ctx.get("bi_encoder_score", ctx["similarity"])
            rr = ctx.get("rerank_score", 0.0)
            preview = ctx["text"][:100] + "..." if len(ctx["text"]) > 100 else ctx["text"]
            print(f"  {i}. bi={bi:.3f} → rerank={rr:.3f}  |  {preview}")
        print()

    # 3. Генерация
    messages = create_rag_messages(query, contexts)
    answer = generate_with_llama(messages, max_new_tokens=max_tokens, temperature=temperature)

    print("─" * 60)
    print(answer)
    print("─" * 60)

    return {"query": query, "contexts": contexts, "answer": answer}


## Бенчмарк скорости

Поиск + reranking + генерация — замерим время и токенов/сек.

In [18]:
test_query = "Расскажи подробно про героя Zeus"

start = time.time()
results = search_all_collections(test_query, top_k=10)
candidates = get_top_contexts(results, top_n=15, min_similarity=0.3)
contexts = rerank_results(test_query, candidates, top_n=5)
search_time = time.time() - start

if not contexts:
    print("Контекст не найден — проверьте, что коллекции проиндексированы")
else:
    messages = create_rag_messages(test_query, contexts)
    # apply_chat_template(tokenize=True) возвращает list[int] напрямую, не dict
    prompt_tokens = len(llama_tokenizer.apply_chat_template(messages, tokenize=True)['input_ids'])
    print(f"Контекстов: {len(contexts)}, лучший rerank={contexts[0]['rerank_score']:.3f}, "
          f"длина промпта: {prompt_tokens} токенов")

    gen_start = time.time()
    answer = generate_with_llama(messages, max_new_tokens=512, temperature=0.2)
    gen_time = time.time() - gen_start

    tokens = len(llama_tokenizer.encode(answer))
    print(f"Поиск+rerank={search_time:.2f}с  Генерация={gen_time:.2f}с  "
          f"Токенов={tokens}  Скорость={tokens/gen_time:.1f} т/с\n")
    print(answer)


Контекстов: 5, лучший rerank=0.772, длина промпта: 1646 токенов
Поиск+rerank=0.17с  Генерация=12.18с  Токенов=422  Скорость=34.7 т/с

Зeus - это сильный дальний бой, который использует молнии для разведки и наносит значительный урон противнику. Его основной целью является дальний бой, что делает его подходящим выбором для игроков, которые предпочитают атаковать с расстояния. 

Его здоровье и мана относительно высокие, что позволяет ему поддерживать долгую борьбу на поле боя. Броня также quite высокая, что помогает ему выжить при наводке противника. Однако его скорость атаки и скорость передвижения не besonders высокие, поэтому он может быть менее эффективен в беге и быстром атаке.

В качестве атрибутов, у него есть способность Thundergod's Wrath, которая наносит урон всем вражеским героям, независимо от их местоположения. Эта способность также раскрывает невидимость вражеских существ вокруг каждого пораженного противника, что делает ее очень полезной для контроля над противником. 

Его

## Что дальше → v2 с LangGraph

Этот пайплайн монолитный: каждый запрос идёт через все три коллекции.  
В **`langgraph_node_agent.ipynb`** реализована v2 — граф состояний с маршрутизацией:

| | v1 (этот ноутбук) | v2 (langgraph_node_agent) |
|---|---|---|
| Поиск | все 3 коллекции всегда | только нужная (router-node) |
| Архитектура | функции + линейный вызов | StateGraph (router → search → rerank → generate) |
| Расширяемость | сложно | добавляй ноды без рефакторинга |


## 🎉 Финальная функция: Интерактивный чат с Dota 2 AI

Создадим простой интерактивный интерфейс для вопросов.

In [19]:
def dota2_chat():
    """
    Интерактивный чат с Dota 2 AI.
    Введите 'exit' или 'quit' для выхода.
    """
    print("Dota 2 AI Assistant — задавайте вопросы о героях, предметах и патчах")
    print("Введите 'exit' для выхода\n")

    while True:
        user_query = input("Вопрос: ").strip()
        if user_query.lower() in ("exit", "quit", "выход"):
            print("До встречи!")
            break
        if not user_query:
            continue
        try:
            ask_dota2_with_llm(user_query, top_k=5, temperature=0.3, show_contexts=False)
            print()
        except Exception as e:
            print(f"{e}")

# Раскомментируйте строку ниже для входа в интерактивный чат
# dota2_chat()